## Introduction

TBD

 ## Python Imports

In [6]:
# Standard Library
import os
import sys
import warnings

from pathlib import Path

# Data Handling
import pandas as pd

# Suppress warnings
warnings.filterwarnings("ignore")

In [7]:
# Add the source subdirectory to the system path to allow import config from settings.py
current_directory = Path(os.getcwd())
website_base_directory = current_directory.parent.parent.parent
src_directory = website_base_directory / "src"
sys.path.append(str(src_directory)) if str(src_directory) not in sys.path else None

# Import settings.py
from settings import config

# Add configured directories from config to path
SOURCE_DIR = config("SOURCE_DIR")
sys.path.append(str(Path(SOURCE_DIR))) if str(Path(SOURCE_DIR)) not in sys.path else None

# Add other configured directories
BASE_DIR = config("BASE_DIR")
CONTENT_DIR = config("CONTENT_DIR")
POSTS_DIR = config("POSTS_DIR")
PAGES_DIR = config("PAGES_DIR")
PUBLIC_DIR = config("PUBLIC_DIR")
SOURCE_DIR = config("SOURCE_DIR")
DATA_DIR = config("DATA_DIR")
DATA_MANUAL_DIR = config("DATA_MANUAL_DIR")

## Python Functions

Here are the functions needed for this project:

* [load_data](/posts/reusable-extensible-python-functions-financial-data-analysis/#load_data): Load data from a CSV, Excel, or Pickle file into a pandas DataFrame.

In [8]:
from load_data import load_data

## Set Variables

In [9]:
ticker = "UXI"
source = "Polygon"
asset_class = "Exchange_Traded_Funds"
timeframe = "day"

## Load Post-Split Data

In [10]:
df_post_split = load_data(
    base_directory=DATA_DIR,
    ticker=f"{ticker}-postsplit",
    source=source,
    asset_class=asset_class,
    timeframe=timeframe,
    file_format="pickle",
)

df_post_split

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2024-07-01 04:00:00,32.1100,32.1102,31.3000,31.3016,1193.000000,31.7245,23,None
1,2024-07-02 04:00:00,31.0000,31.5973,31.0000,31.5973,7607.000000,31.3605,17,None
2,2024-07-03 04:00:00,31.9300,31.9301,31.9194,31.9194,1407.000000,31.9117,29,None
3,2024-07-05 04:00:00,31.6036,31.6036,31.6036,31.6036,205.000000,31.5228,15,None
4,2024-07-08 04:00:00,31.8300,32.1000,31.6674,31.6674,2352.000000,31.7164,14,None
...,...,...,...,...,...,...,...,...,...
494,2026-06-22 04:00:00,61.6700,61.7400,60.7000,61.3905,13323.677969,61.3650,177,None
495,2026-06-23 04:00:00,59.6099,60.0900,58.7800,59.1900,25463.769960,59.8068,193,None
496,2026-06-24 04:00:00,59.2900,61.2500,59.2900,60.2864,7864.116753,60.7548,160,None
497,2026-06-25 04:00:00,62.5500,64.1899,62.2901,62.6300,7086.170590,62.9661,151,None


## Calc Cutoff and Split Check Dates

In [11]:
cutoff_date = df_post_split.loc[3, "Date"]
print(f"Cutoff date for split adjustment: {cutoff_date}")
split_check_date = df_post_split.loc[4, "Date"]
print(f"Split check date: {split_check_date}")

Cutoff date for split adjustment: 2024-07-05 04:00:00
Split check date: 2024-07-08 04:00:00


## Load Pre-Split Data

In [12]:
df_pre_split = load_data(
    base_directory=DATA_DIR,
    ticker=f"{ticker}-presplit",
    source=source,
    asset_class=asset_class,
    timeframe=timeframe,
    file_format="pickle",
)

print(df_pre_split.shape)
df_pre_split[df_pre_split["Date"] < cutoff_date]

(719, 9)


,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-07-31 04:00:00,27.559,27.559,27.559,27.559,283.0,27.535,28,None
1,2023-08-01 04:00:00,27.750,27.789,27.739,27.789,698.0,27.735,14,None
2,2023-08-02 04:00:00,27.280,27.530,27.203,27.203,52597.0,27.529,13,None
3,2023-08-03 04:00:00,26.830,26.881,26.830,26.881,334.0,26.725,14,None
4,2023-08-04 04:00:00,26.759,26.759,26.460,26.460,316.0,26.866,27,None
...,...,...,...,...,...,...,...,...,...
229,2024-06-27 04:00:00,31.902,31.902,31.902,31.902,114.0,31.891,14,None
230,2024-06-28 04:00:00,31.992,31.992,31.992,31.992,40.0,32.106,12,None
231,2024-07-01 04:00:00,32.110,32.110,31.300,31.302,1193.0,31.724,23,None
232,2024-07-02 04:00:00,31.000,31.597,31.000,31.597,7607.0,31.360,17,None


## Calc Split Ratio

In [13]:
# Extract "open" price from the same row from both pre and post split dataframes to calculate the split ratio
pre_split_open_price = df_pre_split[df_pre_split["Date"] == split_check_date]["open"].values[0]
post_split_open_price = df_post_split[df_post_split["Date"] == split_check_date]["open"].values[0]
SPLIT_RATIO = pre_split_open_price / post_split_open_price
print(f"Calculated split ratio: {SPLIT_RATIO}")

Calculated split ratio: 1.0


## Calc Corrected Pre-Split Data

In [14]:
df_pre_split_corrected = df_pre_split.copy()
df_pre_split_corrected["open"] = df_pre_split_corrected["open"] / SPLIT_RATIO
df_pre_split_corrected["high"] = df_pre_split_corrected["high"] / SPLIT_RATIO
df_pre_split_corrected["low"] = df_pre_split_corrected["low"] / SPLIT_RATIO
df_pre_split_corrected["close"] = df_pre_split_corrected["close"] / SPLIT_RATIO
df_pre_split_corrected["volume"] = df_pre_split_corrected["volume"] * SPLIT_RATIO
df_pre_split_corrected["vwap"] = df_pre_split_corrected["vwap"] / SPLIT_RATIO
df_pre_split_corrected[df_pre_split_corrected["Date"] < cutoff_date]

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-07-31 04:00:00,27.559,27.559,27.559,27.559,283.0,27.535,28,None
1,2023-08-01 04:00:00,27.750,27.789,27.739,27.789,698.0,27.735,14,None
2,2023-08-02 04:00:00,27.280,27.530,27.203,27.203,52597.0,27.529,13,None
3,2023-08-03 04:00:00,26.830,26.881,26.830,26.881,334.0,26.725,14,None
4,2023-08-04 04:00:00,26.759,26.759,26.460,26.460,316.0,26.866,27,None
...,...,...,...,...,...,...,...,...,...
229,2024-06-27 04:00:00,31.902,31.902,31.902,31.902,114.0,31.891,14,None
230,2024-06-28 04:00:00,31.992,31.992,31.992,31.992,40.0,32.106,12,None
231,2024-07-01 04:00:00,32.110,32.110,31.300,31.302,1193.0,31.724,23,None
232,2024-07-02 04:00:00,31.000,31.597,31.000,31.597,7607.0,31.360,17,None


## Combine Pre- and Post-Split Data

In [16]:
display_date = df_post_split.loc[5, "Date"]
combined = pd.concat([df_pre_split_corrected[df_pre_split_corrected["Date"] <= cutoff_date], df_post_split], ignore_index=True)
combined[combined["Date"] <= display_date].tail(15)

,Date,open,high,low,close,volume,vwap,transactions,otc
226,2024-06-24 04:00:00,32.6500,33.0500,32.6500,32.7020,1459.0,32.8100,19,None
227,2024-06-25 04:00:00,32.1600,32.1600,31.9140,32.0750,1032.0,32.0470,16,None
228,2024-06-26 04:00:00,31.8200,31.9640,31.6600,31.9640,2614.0,31.8020,46,None
229,2024-06-27 04:00:00,31.9020,31.9020,31.9020,31.9020,114.0,31.8910,14,None
230,2024-06-28 04:00:00,31.9920,31.9920,31.9920,31.9920,40.0,32.1060,12,None
231,2024-07-01 04:00:00,32.1100,32.1100,31.3000,31.3020,1193.0,31.7240,23,None
232,2024-07-02 04:00:00,31.0000,31.5970,31.0000,31.5970,7607.0,31.3600,17,None
233,2024-07-03 04:00:00,31.9300,31.9300,31.9190,31.9190,1407.0,31.9120,29,None
234,2024-07-05 04:00:00,31.6040,31.6040,31.6040,31.6040,205.0,31.5230,15,None
235,2024-07-01 04:00:00,32.1100,32.1102,31.3000,31.3016,1193.0,31.7245,23,None


## Confirm Rows To Be Dropped

In [17]:
print(f"Should drop the following duplicate rows if they exist:")
combined[combined.round(3).duplicated(subset=["Date", "open", "high", "low", "close", "volume"], keep=False)]

Should drop the following duplicate rows if they exist:


,Date,open,high,low,close,volume,vwap,transactions,otc
231,2024-07-01 04:00:00,32.1100,32.1100,31.3000,31.3020,1193.0,31.7240,23,None
232,2024-07-02 04:00:00,31.0000,31.5970,31.0000,31.5970,7607.0,31.3600,17,None
233,2024-07-03 04:00:00,31.9300,31.9300,31.9190,31.9190,1407.0,31.9120,29,None
234,2024-07-05 04:00:00,31.6040,31.6040,31.6040,31.6040,205.0,31.5230,15,None
235,2024-07-01 04:00:00,32.1100,32.1102,31.3000,31.3016,1193.0,31.7245,23,None
236,2024-07-02 04:00:00,31.0000,31.5973,31.0000,31.5973,7607.0,31.3605,17,None
237,2024-07-03 04:00:00,31.9300,31.9301,31.9194,31.9194,1407.0,31.9117,29,None
238,2024-07-05 04:00:00,31.6036,31.6036,31.6036,31.6036,205.0,31.5228,15,None


## Drop Duplicate Rows

In [18]:
combined = pd.concat([df_pre_split_corrected[df_pre_split_corrected["Date"] <= cutoff_date], df_post_split], ignore_index=True).round(3).drop_duplicates(subset=["Date", "open", "high", "low", "close", "volume"], keep="last")
combined[combined["Date"] <= display_date].tail(15)

,Date,open,high,low,close,volume,vwap,transactions,otc
222,2024-06-17 04:00:00,32.150,32.156,32.150,32.156,4282.0,32.129,14,None
223,2024-06-18 04:00:00,32.170,32.473,32.170,32.473,3668.0,32.287,23,None
224,2024-06-20 04:00:00,32.510,32.640,32.500,32.600,5234.0,32.571,68,None
225,2024-06-21 04:00:00,32.390,32.550,32.260,32.550,7566.0,32.336,84,None
226,2024-06-24 04:00:00,32.650,33.050,32.650,32.702,1459.0,32.810,19,None
227,2024-06-25 04:00:00,32.160,32.160,31.914,32.075,1032.0,32.047,16,None
228,2024-06-26 04:00:00,31.820,31.964,31.660,31.964,2614.0,31.802,46,None
229,2024-06-27 04:00:00,31.902,31.902,31.902,31.902,114.0,31.891,14,None
230,2024-06-28 04:00:00,31.992,31.992,31.992,31.992,40.0,32.106,12,None
235,2024-07-01 04:00:00,32.110,32.110,31.300,31.302,1193.0,31.724,23,None


## Reset Index And Display Data

In [19]:
combined.reset_index(drop=True, inplace=True)
combined

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-07-31 04:00:00,27.559,27.559,27.559,27.559,283.000,27.535,28,None
1,2023-08-01 04:00:00,27.750,27.789,27.739,27.789,698.000,27.735,14,None
2,2023-08-02 04:00:00,27.280,27.530,27.203,27.203,52597.000,27.529,13,None
3,2023-08-03 04:00:00,26.830,26.881,26.830,26.881,334.000,26.725,14,None
4,2023-08-04 04:00:00,26.759,26.759,26.460,26.460,316.000,26.866,27,None
...,...,...,...,...,...,...,...,...,...
725,2026-06-22 04:00:00,61.670,61.740,60.700,61.390,13323.678,61.365,177,None
726,2026-06-23 04:00:00,59.610,60.090,58.780,59.190,25463.770,59.807,193,None
727,2026-06-24 04:00:00,59.290,61.250,59.290,60.286,7864.117,60.755,160,None
728,2026-06-25 04:00:00,62.550,64.190,62.290,62.630,7086.171,62.966,151,None


## Export

In [20]:
# Export the combined DataFrame to a pickle file
combined.to_pickle(DATA_DIR / source / asset_class / timeframe / f"{ticker}.pkl")